In [ ]:
import pandas as pd
from transformers import pipeline
import os  

print("="*60)
print("KR-FinBERT 로드 중 (사업보고서)")
finbert = pipeline("text-classification", model="snunlp/KR-FinBert-SC", device=-1)
print("="*60)

# 긍정/부정 결과를 0.0 ~ 1.0 사이의 수치로 나타냄
def get_sentiment_score(text):
    try:
        # KR-FinBert 모델의 최대 토큰이 512이므로 안전장치 적용
        result = finbert(text, truncation=True, max_length=512)[0]
        label = result['label'].lower()
        score = result['score'] 
        
        # 긍정일 경우(예: 긍정 0.9 -> 0.9)
        if 'pos' in label or '1' in label:
            result_score = score
        # 부정일 경우: (예: 부정 0.9 -> 0.1)
        elif 'neg' in label or '0' in label:
            result_score = 1.0 - score
        # 중립일 경우: 0.5 
        else:
            return 0.5
        
        return round(result_score, 1)
    
    except Exception as e: # 예외처리도 중립
        return 0.5 

input_folder = "../data/Report"
output_folder = "../data/Report_Positive"

# 가져온 csv 데이터 파일 보고서 긍정값
csv_files = [
    "Hanmi_Semiconductor_2020_2024.csv",
    "Samsung_Electronics_Comprehensive_2020_2024.csv",
    "SK_Hynix_Comprehensive_2020_2024.csv"
]


for file_name in csv_files:
    print(f"\n [{file_name}] 파일 탐색")

    input_filepath = os.path.join(input_folder, file_name)

    try:
        df = pd.read_csv(input_filepath)

        print("감성 분석 중")
        df['보고서_긍정값'] = df['Unstructured_Text_Content'].apply(get_sentiment_score)
        #data/report_positive 폴더 안에 csv 생성
        output_filename = f"{file_name}"
        output_filepath = os.path.join(output_folder, output_filename)

        # 생성한 경로에 파일 저장
        df.to_csv(output_filepath, index=False, encoding='utf-8-sig')
        print(f" '{output_filepath}' 경로에 저장되었습니다.")
        
    except Exception as e:
        print(f" 에러 발생: {e}")

KR-FinBERT 로드 중 (뉴스/사업보고서 전용)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 66106.41it/s]



 [Hanmi_Semiconductor_2020_2024.csv] 파일 탐색
감성 분석 중
 '../data/Report_Positive\Hanmi_Semiconductor_2020_2024.csv' 경로에 안전하게 저장되었습니다.

 [Samsung_Electronics_Comprehensive_2020_2024.csv] 파일 탐색
감성 분석 중
 '../data/Report_Positive\Samsung_Electronics_Comprehensive_2020_2024.csv' 경로에 안전하게 저장되었습니다.

 [SK_Hynix_Comprehensive_2020_2024.csv] 파일 탐색
감성 분석 중
 '../data/Report_Positive\SK_Hynix_Comprehensive_2020_2024.csv' 경로에 안전하게 저장되었습니다.


: 

In [4]:
import pandas as pd
import numpy as np
from transformers import pipeline
import os  

print("="*60)
print("KR-FinBERT 로드 중 (뉴스보고서)")
finbert = pipeline("text-classification", model="snunlp/KR-FinBert-SC", device=-1)
print("="*60)

# 긍정/부정 결과를 0.0 ~ 1.0 사이의 수치로 나타냄
def get_sentiment_score(text):
    try:
        # KR-FinBert 모델의 최대 토큰이 512이므로 안전장치 적용
        result = finbert(text, truncation=True, max_length=512)[0]
        label = result['label'].lower()
        score = result['score'] 
        
        # 긍정일 경우(예: 긍정 0.9 -> 0.9)
        if 'pos' in label or '1' in label:
            result_score = score
        # 부정일 경우: (예: 부정 0.9 -> 0.1)
        elif 'neg' in label or '0' in label:
            result_score = 1.0 - score
        # 중립일 경우: 0.5 
        else:
            return 0.5
        
        return round(result_score, 1)
    
    except Exception as e: # 예외처리도 중립
        return 0.5 
#본문 문장 뽑아서 평균값 내기
def get_avg_sentiment(text):
    #1.결측치 예외 처리
    if not isinstance(text, str) or not text.strip():
        return 0.5
    
    # 2. 쉼표를 기준으로 분리하고 양쪽 공백 제거
    segments = [s.strip() for s in text.split(',')]

    segments = [s for s in segments if len(s) > 0]
    
    if not segments:
        return 0.5

    #3. 분리된 모든 조각을 전부 분석하여 점수 수집
    scores = []
    for segment in segments:
        scores.append(get_sentiment_score(segment))

    # 4. 수집된 긍정값들의 평균을 계산 
    return round(np.mean(scores), 2)


    
input_folder = "../data/news_data"
output_folder = "../data/news_Positive"

# 가져온 csv 데이터 파일 보고서 긍정값
csv_files = [
    "bigkinds_semiconductor_news_20260514_20260603_raw.csv"
]


for file_name in csv_files:
    print(f"\n [{file_name}] 파일 탐색")

    input_filepath = os.path.join(input_folder, file_name)

    try:
        df = pd.read_csv(input_filepath)

        print("감성 분석 중")
        df['뉴스_긍정값'] = df['본문'].astype(str).apply(get_sentiment_score)
        if '분석제외 여부' in df.columns:
            df = df[df['분석제외 여부'].isna() | (df['분석제외 여부'] == '')]
        
        # '특성추출' 또는 '키워드' 컬럼을 타겟으로 지정
        target_column = '특성추출(가중치순 상위 50개)'
        
        if target_column in df.columns:
            df['뉴스_긍정값'] = df[target_column].apply(get_avg_sentiment)
        else:
            print(f"데이터에 '{target_column}' 컬럼이 없어 '본문'으로 대체합니다.")
            df['뉴스_긍정값'] = df['본문'].apply(get_avg_sentiment)



        #data/report_positive 폴더 안에 csv 생성
        output_filename = f"{file_name}"
        output_filepath = os.path.join(output_folder, output_filename)

        # 생성한 경로에 파일 저장
        df.to_csv(output_filepath, index=False, encoding='utf-8-sig')
        print(f" '{output_filepath}' 경로에 저장되었습니다.")
       
        
    except Exception as e:
        print(f" 에러 발생: {e}")

KR-FinBERT 로드 중 (뉴스보고서)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 41995.27it/s]



 [bigkinds_semiconductor_news_20260514_20260603_raw.csv] 파일 탐색
감성 분석 중
 '../data/news_Positive\bigkinds_semiconductor_news_20260514_20260603_raw.csv' 경로에 저장되었습니다.
